12_ibdb_cast_extraction.ipynb

01 imports
02 load production catalogue
03 define helper functions
04 test on Oklahoma
05 scrape all productions
06 save final dataset

In [21]:
import requests
from bs4 import BeautifulSoup
import re
import pandas as pd
import time

from tqdm import tqdm

In [22]:
import pandas as pd


production_catalogue = pd.read_csv(
    "../data/processed/production_catalogue.csv"
)


production_catalogue_unique = (
    production_catalogue
    .drop_duplicates(
        subset="production_id"
    )
    .reset_index(drop=True)
)


print(production_catalogue_unique.shape)

production_catalogue_unique.to_csv(
    "../data/processed/production_catalogue_unique.csv",
    index=False
)

(7292, 6)


In [23]:
production_catalogue = pd.read_csv(
    "../data/processed/production_catalogue_unique.csv"
)

production_catalogue.head()

,production_id,title,url,season_id,season_label,start_year
0,10336,Bottomland,https://www.ibdb.com/broadway-production/botto...,1029,1927-1928,1927
1,10337,Padlocks of 1927,https://www.ibdb.com/broadway-production/padlo...,1029,1927-1928,1927
2,10338,Madame X,https://www.ibdb.com/broadway-production/madam...,1029,1927-1928,1927
3,444423,Africana [1927],https://www.ibdb.com/broadway-production/afric...,1029,1927-1928,1927
4,10340,Rang Tang,https://www.ibdb.com/broadway-production/rang-...,1029,1927-1928,1927


In [24]:
headers = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    ),
    "Accept": (
        "text/html,application/xhtml+xml,application/xml;"
        "q=0.9,image/avif,image/webp,*/*;q=0.8"
    ),
    "Accept-Language": "en-US,en;q=0.9",
    "Accept-Encoding": "gzip, deflate",
    "Connection": "keep-alive",
}

In [25]:
def extract_performer_id(url):
    """
    Extract IBDB performer ID from URL.

    Example:
    /broadway-cast-staff/alfred-drake-4031 -> 4031
    """
    
    return url.split("-")[-1]

In [26]:
def extract_cast(url, session):
    """
    Extract Opening Night Cast from an IBDB production page.

    Returns:
        list of dictionaries containing:
        performer_id,
        performer_name,
        character
    """

    response = session.get(
        url,
        headers=headers
    )

    response.raise_for_status()


    soup = BeautifulSoup(
        response.text,
        "html.parser"
    )


    cast_section = soup.find(
        id="OpeningNightCast"
    )


    if cast_section is None:
        raise ValueError(
            "Opening Night Cast section not found"
        )


    cast_rows = cast_section.find_all(
        "div",
        class_="row mobile-a-align"
    )


    cast = []


    for row in cast_rows:

        performer = row.find(
            "a",
            href=re.compile(
                "/broadway-cast-staff/"
            )
        )


        if performer is None:
            continue


        columns = row.find_all(
            "div",
            class_="col m4 s12"
        )


        character = None


        if len(columns) > 1:
            character = columns[1].get_text(
                strip=True
            )


        cast.append(
            {
                "performer_id": extract_performer_id(
                    performer["href"]
                ),
                "performer_name": performer.text.strip(),
                "character": character
            }
        )


    return cast

In [27]:
session = requests.Session()

cast_records = []


for i, row in tqdm(
    production_catalogue.iterrows(),
    total=len(production_catalogue)
):

    retries = 3


    for attempt in range(retries):

        try:

            cast = extract_cast(
                row["url"],
                session
            )


            for performer in cast:

                performer["production_id"] = (
                    row["production_id"]
                )

                cast_records.append(
                    performer
                )


            break


        except Exception as e:

            print(
                f"Error scraping {row['production_id']} "
                f"(attempt {attempt + 1}/{retries}): {e}"
            )


            if attempt < retries - 1:
                time.sleep(5)


            else:
                print(
                    f"FAILED: {row['production_id']}"
                )


    # checkpoint every 100 productions

    if i % 100 == 0:

        pd.DataFrame(
            cast_records
        ).to_csv(
            "../data/processed/production_cast_checkpoint.csv",
            index=False
        )


    # polite delay

    time.sleep(0.3)

 94%|█████████▍| 6854/7292 [2:15:12<09:30,  1.30s/it]

Error scraping 496692 (attempt 1/3): 429 Client Error: Too Many Requests for url: https://www.ibdb.com/broadway-production/the-country-house-496692


100%|█████████▉| 7282/7292 [2:25:03<00:10,  1.04s/it]

Error scraping 547118 (attempt 1/3): Opening Night Cast section not found
Error scraping 547118 (attempt 2/3): Opening Night Cast section not found
Error scraping 547118 (attempt 3/3): Opening Night Cast section not found
FAILED: 547118


100%|█████████▉| 7287/7292 [2:25:16<00:06,  1.34s/it]

...

100%|█████████▉| 7289/7292 [2:25:27<00:09,  3.02s/it]

Error scraping 547120 (attempt 1/3): Opening Night Cast section not found
Error scraping 547120 (attempt 2/3): Opening Night Cast section not found
Error scraping 547120 (attempt 3/3): Opening Night Cast section not found
FAILED: 547120


100%|█████████▉| 7291/7292 [2:25:38<00:03,  3.84s/it]

Error scraping 547328 (attempt 1/3): Opening Night Cast section not found
Error scraping 547328 (attempt 2/3): Opening Night Cast section not found
Error scraping 547328 (attempt 3/3): Opening Night Cast section not found
FAILED: 547328


100%|██████████| 7292/7292 [2:25:49<00:00,  1.20s/it]


In [28]:
production_cast = pd.DataFrame(
    cast_records
)


production_cast.head()

,performer_id,performer_name,character,production_id
0,473388,Dot Campbell,Chorus,10336
1,34408,Raymond Campbell,Skinny,10336
2,473389,Alice Carter,Chorus,10336
3,35813,Louis Cole,Jimmy,10336
4,438924,Craddock and Shadney,Specialty,10336


In [29]:
production_cast.shape

(115677, 4)

In [30]:
production_cast.to_csv(
    "../data/processed/production_cast.csv",
    index=False
)